In [ ]:
import os
import openai
import warnings
warnings.filterwarnings('ignore')

os.environ["OPENAI_API_KEY"] = "sk-llave-secreta-de-openai-la-que-no-tengo"
openai.api_key = os.environ["OPENAI_API_KEY"]

from llama_index import SimpleDirectoryReader, Document
from llama_index.llms import OpenAI

documents = SimpleDirectoryReader(
    input_files=["./ArquitecturaLLM.pdf"]
).load_data()

document = Document(text="\n\n".join([doc.text for doc in documents]))

llm = OpenAI(model="gpt-3.5-turbo", temperature=0.1)

In [ ]:
from llama_index import ServiceContext, VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.node_parser import SentenceWindowNodeParser
from llama_index.indices.postprocessor import MetadataReplacementPostProcessor, SentenceTransformerRerank

def setup_sentence_window(docs, llm, save_dir="sentence_index"):
    node_parser = SentenceWindowNodeParser.from_defaults(
        window_size=3,
        window_metadata_key="window",
        original_text_metadata_key="original_text",
    )
    
    sentence_context = ServiceContext.from_defaults(
        llm=llm,
        embed_model="local:BAAI/bge-small-en-v1.5",
        node_parser=node_parser,
    )
    
    if not os.path.exists(save_dir):
        index = VectorStoreIndex.from_documents(docs, service_context=sentence_context)
        index.storage_context.persist(persist_dir=save_dir)
    else:
        index = load_index_from_storage(
            StorageContext.from_defaults(persist_dir=save_dir),
            service_context=sentence_context,
        )
    return index

sentence_index = setup_sentence_window([document], llm)

postproc = MetadataReplacementPostProcessor(target_metadata_key="window")
rerank = SentenceTransformerRerank(top_n=2, model="BAAI/bge-reranker-base")

sentence_engine = sentence_index.as_query_engine(
    similarity_top_k=6, node_postprocessors=[postproc, rerank]
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9902.80it/s]


In [ ]:
from llama_index.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.retrievers import AutoMergingRetriever
from llama_index.query_engine import RetrieverQueryEngine

def setup_automerging(docs, llm, save_dir="merging_index"):
    node_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=[2048, 512, 128])
    nodes = node_parser.get_nodes_from_documents(docs)
    leaf_nodes = get_leaf_nodes(nodes)
    
    merging_context = ServiceContext.from_defaults(
        llm=llm,
        embed_model="local:BAAI/bge-small-en-v1.5",
    )
    
    storage_context = StorageContext.from_defaults()
    storage_context.docstore.add_documents(nodes)
    
    if not os.path.exists(save_dir):
        index = VectorStoreIndex(leaf_nodes, storage_context=storage_context, service_context=merging_context)
        index.storage_context.persist(persist_dir=save_dir)
    else:
        index = load_index_from_storage(
            StorageContext.from_defaults(persist_dir=save_dir),
            service_context=merging_context,
        )
    return index

automerge_index = setup_automerging([document], llm)

base_retriever = automerge_index.as_retriever(similarity_top_k=12)
retriever = AutoMergingRetriever(base_retriever, automerge_index.storage_context, verbose=True)

automerge_engine = RetrieverQueryEngine.from_args(
    retriever, node_postprocessors=[rerank]
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11000.55it/s]


In [ ]:
# Pregunta de prueba
query = "What is the General Architecture of an LLM?"

print("============== ESTRATEGIA 1: SENTENCE WINDOW ==============\n")
response_1 = sentence_engine.query(query)
print(str(response_1))
print("\n-----------------------------------------------------------\n")

print("============== ESTRATEGIA 2: AUTO-MERGING ==============\n")
response_2 = automerge_engine.query(query)
print(str(response_2))

============== ESTRATEGIA 1: SENTENCE WINDOW ==============



AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-tu-ll**********************aqui. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}